# Customer Churn Prediction

Проект бинарной классификации: модель должна предсказывать, уйдёт ли клиент из банка.

Цель проекта - не только получить качественные предсказания, но и определить клиентов с высоким риском ухода, чтобы банк мог заранее предложить им меры удержания.

In [2]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_openml

RANDOM_STATE = 42

X, y = fetch_openml(
    data_id=46911,
    as_frame=True,
    return_X_y=True,
    parser="auto",
)

print("Размер X:", X.shape)
print("Размер y:", y.shape)
print("\nТипы признаков:")
print(X.dtypes.value_counts())

print("\nКоличество пропусков:", X.isna().sum().sum())
print("\nРаспределение целевого класса:")
print(y.value_counts(dropna=False))

display(X.head())

Размер X: (10000, 10)
Размер y: (10000,)

Типы признаков:
int64       4
category    2
float64     2
category    1
category    1
Name: count, dtype: int64

Количество пропусков: 0

Распределение целевого класса:
churn
No     7963
Yes    2037
Name: count, dtype: int64


,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
0,596,Germany,Male,32,3,96709.07,2,0,0,41788.37
1,623,France,Male,43,1,0.00,2,1,1,146379.30
2,601,Spain,Female,44,4,0.00,2,1,0,58561.31
3,506,Germany,Male,59,8,119152.10,2,1,1,170679.74
4,560,Spain,Female,27,7,124995.98,1,1,1,114669.79


In [3]:
from sklearn.model_selection import train_test_split

# Yes - клиент ушёл, это положительный класс
y_binary = y.map({"No": 0, "Yes": 1}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_binary,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_binary,
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("\nДоля ушедших в train:", y_train.mean().round(4))
print("Доля ушедших в test:", y_test.mean().round(4))

Train: (8000, 10)
Test: (2000, 10)

Доля ушедших в train: 0.2037
Доля ушедших в test: 0.2035


In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

dummy = DummyClassifier(strategy="most_frequent")

dummy_cv = cross_validate(
    dummy,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
)

for metric in scoring:
    mean = dummy_cv[f"test_{metric}"].mean()
    std = dummy_cv[f"test_{metric}"].std()

    print(f"Dummy CV {metric}: {mean:.3f} ± {std:.3f}")

Dummy CV accuracy: 0.796 ± 0.000
Dummy CV precision: 0.000 ± 0.000
Dummy CV recall: 0.000 ± 0.000
Dummy CV f1: 0.000 ± 0.000
Dummy CV roc_auc: 0.500 ± 0.000
Dummy CV pr_auc: 0.204 ± 0.000


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_features = X_train.select_dtypes(include="number").columns
categorical_features = X_train.select_dtypes(exclude="number").columns

print("Числовые признаки:", list(numeric_features))
print("Категориальные признаки:", list(categorical_features))

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE,
        ),
    ),
])

logistic_cv = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
)

for metric in scoring:
    mean = logistic_cv[f"test_{metric}"].mean()
    std = logistic_cv[f"test_{metric}"].std()

    print(f"Logistic CV {metric}: {mean:.3f} ± {std:.3f}")

Числовые признаки: ['credit_score', 'age', 'tenure', 'balance', 'products_number', 'estimated_salary']
Категориальные признаки: ['country', 'gender', 'credit_card', 'active_member']
Logistic CV accuracy: 0.811 ± 0.005
Logistic CV precision: 0.597 ± 0.035
Logistic CV recall: 0.225 ± 0.017
Logistic CV f1: 0.326 ± 0.020
Logistic CV roc_auc: 0.768 ± 0.013
Logistic CV pr_auc: 0.472 ± 0.018


In [6]:
from sklearn.base import clone

balanced_logistic_model = clone(logistic_model)

balanced_logistic_model.set_params(
    model__class_weight="balanced"
)

balanced_logistic_cv = cross_validate(
    balanced_logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
)

comparison = pd.DataFrame(
    {
        "Logistic": [
            logistic_cv[f"test_{metric}"].mean()
            for metric in scoring
        ],
        "Logistic balanced": [
            balanced_logistic_cv[f"test_{metric}"].mean()
            for metric in scoring
        ],
    },
    index=scoring.keys(),
).round(3)

display(comparison)

,Logistic,Logistic balanced
accuracy,0.811,0.714
precision,0.597,0.387
recall,0.225,0.689
f1,0.326,0.496
roc_auc,0.768,0.771
pr_auc,0.472,0.461


In [7]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

oof_proba = cross_val_predict(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

threshold_rows = []

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    oof_pred = (oof_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_train,
        oof_pred,
    ).ravel()

    threshold_rows.append({
        "threshold": threshold,
        "contact_rate": oof_pred.mean(),
        "precision": precision_score(y_train, oof_pred),
        "recall": recall_score(y_train, oof_pred),
        "f1": f1_score(y_train, oof_pred),
        "FP": fp,
        "FN": fn,
    })

threshold_results = pd.DataFrame(threshold_rows)

display(
    threshold_results.style.format({
        "threshold": "{:.2f}",
        "contact_rate": "{:.1%}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1": "{:.3f}",
    })
)

,threshold,contact_rate,precision,recall,f1,FP,FN
0,0.10,67.5%,0.273,0.904,0.419,3925,157
1,0.20,37.9%,0.370,0.688,0.481,1908,509
2,0.30,21.6%,0.473,0.501,0.487,908,814
3,0.40,12.8%,0.541,0.339,0.416,469,1078
4,0.50,7.7%,0.596,0.225,0.326,248,1264


In [8]:
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=5,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
])

gradient_boosting_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=15,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=RANDOM_STATE,
        ),
    ),
])

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model,
}

model_rows = []

for model_name, model in models.items():
    result = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1,
    )

    model_rows.append({
        "model": model_name,
        "cv_precision": result["test_precision"].mean(),
        "cv_recall": result["test_recall"].mean(),
        "cv_f1": result["test_f1"].mean(),
        "cv_roc_auc": result["test_roc_auc"].mean(),
        "cv_pr_auc": result["test_pr_auc"].mean(),
        "cv_pr_auc_std": result["test_pr_auc"].std(),
        "train_pr_auc": result["train_pr_auc"].mean(),
    })

model_comparison = (
    pd.DataFrame(model_rows)
    .sort_values("cv_pr_auc", ascending=False)
    .reset_index(drop=True)
)

display(model_comparison.round(3))

,model,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,cv_pr_auc_std,train_pr_auc
0,Gradient Boosting,0.737,0.487,0.586,0.861,0.697,0.015,0.849
1,Random Forest,0.782,0.416,0.543,0.855,0.675,0.020,0.894
2,Logistic Regression,0.597,0.225,0.326,0.768,0.472,0.018,0.472


In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__max_iter": [200, 300, 500],
    "model__learning_rate": [0.03, 0.05],
    "model__max_leaf_nodes": [7, 15],
    "model__min_samples_leaf": [20, 40],
}

gb_search = GridSearchCV(
    estimator=gradient_boosting_model,
    param_grid=param_grid,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

gb_search.fit(X_train, y_train)

gb_results = pd.DataFrame(gb_search.cv_results_)

gb_results["pr_auc_gap"] = (
    gb_results["mean_train_pr_auc"]
    - gb_results["mean_test_pr_auc"]
)

columns = [
    "param_model__max_iter",
    "param_model__learning_rate",
    "param_model__max_leaf_nodes",
    "param_model__min_samples_leaf",
    "mean_train_pr_auc",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_f1",
    "mean_test_roc_auc",
    "mean_test_pr_auc",
    "std_test_pr_auc",
    "pr_auc_gap",
]

display(
    gb_results[columns]
    .sort_values("mean_test_pr_auc", ascending=False)
    .head(10)
    .round(3)
)

print("Лучшие параметры:")
print(gb_search.best_params_)

print(
    "Лучший CV PR-AUC:",
    round(gb_search.best_score_, 3),
)

,param_model__max_iter,param_model__learning_rate,param_model__max_leaf_nodes,param_model__min_samples_leaf,mean_train_pr_auc,mean_test_precision,mean_test_recall,mean_test_f1,mean_test_roc_auc,mean_test_pr_auc,std_test_pr_auc,pr_auc_gap
16,300,0.05,7,20,0.771,0.755,0.482,0.588,0.866,0.703,0.017,0.068
8,500,0.03,7,20,0.771,0.752,0.480,0.586,0.865,0.702,0.017,0.068
20,500,0.05,7,20,0.803,0.748,0.480,0.584,0.865,0.702,0.015,0.101
2,200,0.03,15,20,0.777,0.759,0.477,0.586,0.865,0.702,0.019,0.076
9,500,0.03,7,40,0.767,0.750,0.481,0.586,0.865,0.701,0.017,0.066
12,200,0.05,7,20,0.750,0.755,0.475,0.583,0.865,0.701,0.017,0.049
6,300,0.03,15,20,0.805,0.742,0.485,0.586,0.863,0.701,0.015,0.103
17,300,0.05,7,40,0.768,0.756,0.483,0.589,0.864,0.701,0.017,0.067
4,300,0.03,7,20,0.745,0.755,0.470,0.578,0.865,0.701,0.017,0.044
21,500,0.05,7,40,0.799,0.744,0.482,0.584,0.863,0.700,0.015,0.098


Лучшие параметры:
{'model__learning_rate': 0.05, 'model__max_iter': 300, 'model__max_leaf_nodes': 7, 'model__min_samples_leaf': 20}
Лучший CV PR-AUC: 0.703


In [10]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

tuned_gb_model = gb_search.best_estimator_

gb_oof_proba = cross_val_predict(
    tuned_gb_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

def evaluate_top_fraction(
    y_true,
    probabilities,
    contact_fraction=0.25,
):
    threshold = np.quantile(
        probabilities,
        1 - contact_fraction,
    )

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
    ).ravel()

    return {
        "threshold": threshold,
        "contact_rate": predictions.mean(),
        "precision": precision_score(y_true, predictions),
        "recall": recall_score(y_true, predictions),
        "f1": f1_score(y_true, predictions),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(
            y_true,
            probabilities,
        ),
        "FP": fp,
        "FN": fn,
    }

budget_comparison = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            **evaluate_top_fraction(
                y_train,
                oof_proba,
                contact_fraction=0.25,
            ),
        },
        {
            "model": "Gradient Boosting",
            **evaluate_top_fraction(
                y_train,
                gb_oof_proba,
                contact_fraction=0.25,
            ),
        },
    ]
)

display(
    budget_comparison.round({
        "threshold": 3,
        "contact_rate": 3,
        "precision": 3,
        "recall": 3,
        "f1": 3,
        "roc_auc": 3,
        "pr_auc": 3,
    })
)

,model,threshold,contact_rate,precision,recall,f1,roc_auc,pr_auc,FP,FN
0,Logistic Regression,0.274,0.25,0.452,0.554,0.498,0.767,0.468,1097,727
1,Gradient Boosting,0.252,0.25,0.566,0.695,0.624,0.865,0.701,867,497


In [11]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss

probability_quality = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "brier_score": brier_score_loss(
            y_train,
            oof_proba,
        ),
        "log_loss": log_loss(
            y_train,
            oof_proba,
        ),
    },
    {
        "model": "Gradient Boosting",
        "brier_score": brier_score_loss(
            y_train,
            gb_oof_proba,
        ),
        "log_loss": log_loss(
            y_train,
            gb_oof_proba,
        ),
    },
])

display(probability_quality.round(4))

actual_rate, predicted_probability = calibration_curve(
    y_train,
    gb_oof_proba,
    n_bins=10,
    strategy="quantile",
)

calibration_table = pd.DataFrame({
    "group": range(1, len(actual_rate) + 1),
    "predicted_probability": predicted_probability,
    "actual_churn_rate": actual_rate,
})

calibration_table["difference"] = (
    calibration_table["predicted_probability"]
    - calibration_table["actual_churn_rate"]
)

display(calibration_table.round(3))

,model,brier_score,log_loss
0,Logistic Regression,0.1356,0.4284
1,Gradient Boosting,0.1019,0.3347


,group,predicted_probability,actual_churn_rate,difference
0,1,0.016,0.018,-0.001
1,2,0.028,0.022,0.006
2,3,0.042,0.034,0.008
3,4,0.061,0.066,-0.006
4,5,0.084,0.076,0.007
5,6,0.117,0.126,-0.009
6,7,0.167,0.172,-0.005
7,8,0.256,0.252,0.004
8,9,0.446,0.458,-0.011
9,10,0.811,0.812,-0.001


In [13]:
from sklearn.inspection import permutation_importance

analysis_train_idx, analysis_val_idx = next(
    cv.split(X_train, y_train)
)

X_analysis_train = X_train.iloc[analysis_train_idx]
X_analysis_val = X_train.iloc[analysis_val_idx]

y_analysis_train = y_train.iloc[analysis_train_idx]
y_analysis_val = y_train.iloc[analysis_val_idx]

importance_model = clone(tuned_gb_model)

importance_model.fit(
    X_analysis_train,
    y_analysis_train,
)

permutation_result = permutation_importance(
    importance_model,
    X_analysis_val,
    y_analysis_val,
    scoring="average_precision",
    n_repeats=15,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_table = (
    pd.DataFrame({
        "feature": X_train.columns,
        "pr_auc_decrease": permutation_result.importances_mean,
        "importance_std": permutation_result.importances_std,
    })
    .sort_values(
        "pr_auc_decrease",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(importance_table.round(4))

,feature,pr_auc_decrease,importance_std
0,age,0.2533,0.0167
1,products_number,0.2320,0.0143
2,active_member,0.0884,0.0138
3,balance,0.0458,0.0062
4,country,0.0414,0.0059
5,gender,0.0132,0.0029
6,credit_score,0.0042,0.0023
7,estimated_salary,0.0009,0.0030
8,credit_card,0.0002,0.0002
9,tenure,-0.0012,0.0023


In [14]:
analysis_df = X_train.copy()

analysis_df["churn"] = y_train
analysis_df["predicted_probability"] = gb_oof_proba

analysis_df["age_group"] = pd.cut(
    analysis_df["age"],
    bins=[0, 30, 40, 50, 60, np.inf],
    labels=[
        "до 30",
        "30-39",
        "40-49",
        "50-59",
        "60+",
    ],
    right=False,
)

age_analysis = (
    analysis_df
    .groupby("age_group", observed=True)
    .agg(
        clients=("churn", "size"),
        actual_churn_rate=("churn", "mean"),
        predicted_probability=(
            "predicted_probability",
            "mean",
        ),
    )
)

products_analysis = (
    analysis_df
    .groupby("products_number", observed=True)
    .agg(
        clients=("churn", "size"),
        actual_churn_rate=("churn", "mean"),
        predicted_probability=(
            "predicted_probability",
            "mean",
        ),
    )
)

activity_analysis = (
    analysis_df
    .groupby("active_member", observed=True)
    .agg(
        clients=("churn", "size"),
        actual_churn_rate=("churn", "mean"),
        predicted_probability=(
            "predicted_probability",
            "mean",
        ),
    )
)

print("Возраст:")
display(age_analysis.round(3))

print("Количество продуктов:")
display(products_analysis.round(3))

print("Активность клиента:")
display(activity_analysis.round(3))

Возраст:


,clients,actual_churn_rate,predicted_probability
age_group,,,
до 30,1309,0.077,0.079
30-39,3478,0.106,0.104
40-49,2086,0.307,0.308
50-59,708,0.561,0.549
60+,419,0.296,0.296


Количество продуктов:


,clients,actual_churn_rate,predicted_probability
products_number,,,
1,4088,0.277,0.274
2,3657,0.077,0.080
3,201,0.796,0.779
4,54,1.000,0.971


Активность клиента:


,clients,actual_churn_rate,predicted_probability
active_member,,,
0,3889,0.267,0.264
1,4111,0.144,0.145


In [15]:
from sklearn.metrics import accuracy_score

FINAL_THRESHOLD = float(
    budget_comparison.loc[
        budget_comparison["model"] == "Gradient Boosting",
        "threshold",
    ].iloc[0]
)

final_model = clone(tuned_gb_model)
final_model.fit(X_train, y_train)

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= FINAL_THRESHOLD).astype(int)

dummy_final = DummyClassifier(strategy="prior")
dummy_final.fit(X_train, y_train)

dummy_test_pred = dummy_final.predict(X_test)
dummy_test_proba = dummy_final.predict_proba(X_test)[:, 1]

def calculate_test_metrics(
    model_name,
    y_true,
    predictions,
    probabilities,
):
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(y_true, predictions),
        "f1": f1_score(y_true, predictions),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(
            y_true,
            probabilities,
        ),
        "brier_score": brier_score_loss(
            y_true,
            probabilities,
        ),
        "contact_rate": predictions.mean(),
    }

test_results = pd.DataFrame([
    calculate_test_metrics(
        "Dummy",
        y_test,
        dummy_test_pred,
        dummy_test_proba,
    ),
    calculate_test_metrics(
        "Gradient Boosting",
        y_test,
        test_pred,
        test_proba,
    ),
])

display(test_results.round(3))

print("Финальный порог:", round(FINAL_THRESHOLD, 3))
print("\nМатрица ошибок Gradient Boosting:")
print(confusion_matrix(y_test, test_pred))

,model,accuracy,precision,recall,f1,roc_auc,pr_auc,brier_score,contact_rate
0,Dummy,0.796,0.000,0.00,0.000,0.500,0.204,0.162,0.000
1,Gradient Boosting,0.828,0.562,0.71,0.628,0.877,0.730,0.098,0.257


Финальный порог: 0.252

Матрица ошибок Gradient Boosting:
[[1368  225]
 [ 118  289]]


## Финальный результат

Для итоговой модели выбран Gradient Boosting. Модель обучалась только на
обучающей выборке, а порог 0.252 был выбран по out-of-fold прогнозам при
бизнес-ограничении на контакт примерно с 25% клиентов.

На отложенной тестовой выборке модель получила:

- PR-AUC: 0.730
- ROC-AUC: 0.877
- Recall: 0.710
- Precision: 0.562
- F1: 0.628
- Brier score: 0.098
- Contact rate: 25.7%

Модель обнаружила 289 из 407 уходящих клиентов. Основными признаками для
ранжирования оказались возраст, количество продуктов, активность клиента,
баланс и страна. Найденные зависимости являются ассоциациями и не
доказывают причинное влияние признаков на уход.

In [16]:
import joblib
import sklearn

MODEL_PATH = "customer_churn_model.joblib"

model_bundle = {
    "model": final_model,
    "threshold": FINAL_THRESHOLD,
    "contact_limit": 0.25,
    "feature_names": list(X_train.columns),
    "positive_class": 1,
    "sklearn_version": sklearn.__version__,
}

joblib.dump(model_bundle, MODEL_PATH)

loaded_bundle = joblib.load(MODEL_PATH)

loaded_proba = (
    loaded_bundle["model"]
    .predict_proba(X_test)[:, 1]
)

print("Файл создан:", MODEL_PATH)
print("Версия sklearn:", sklearn.__version__)
print(
    "Прогнозы совпадают:",
    np.allclose(test_proba, loaded_proba),
)
print(
    "Порог совпадает:",
    loaded_bundle["threshold"] == FINAL_THRESHOLD,
)

Файл создан: customer_churn_model.joblib
Версия sklearn: 1.6.1
Прогнозы совпадают: True
Порог совпадает: True


In [17]:
from google.colab import files

files.download("customer_churn_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>